In [ ]:
################################################################################
#   Código OUT1 - Exemplo da transformação de variáveis Seção 2.3
################################################################################
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Gerar 1000 números aleatórios de uma distribuição Chi-quadrada com 2 graus de liberdade
np.random.seed(18) # Para reprodutibilidade
chi2_data = np.random.chisquare(df=2, size=1000)

# 2. Aplicar transformação de raiz quadrada
sqrt_transformed_data = np.sqrt(chi2_data)

# 3. Plotar dois histogramas separados (original e raiz quadrada)
fig, axes = plt.subplots(1, 2, figsize=(12, 5)) # Alterado para 2 subplots

# Histograma para os dados Chi-quadrados originais
sns.histplot(chi2_data, kde=False, ax=axes[0], color='lightgray')
axes[0].set_title('Histograma dos Dados Originais')
axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Frequência')

# Histograma para os dados raiz-quadrada-transformados
sns.histplot(sqrt_transformed_data, kde=False, ax=axes[1], color='lightgray')
axes[1].set_title('Histograma dos Dados Transformados')
axes[1].set_xlabel('Valor (escala raiz quadrada)')
axes[1].set_ylabel('Frequência')

# Ajustar layout e exibir gráficos
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import itertools # Import para combinações
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pyod.models.iforest import IForest

# --- Replicating data loading and iForest detection from OUT15 to ensure df and selected_features are available ---
# 1. Carregar os dados do arquivo World Happiness Report 2019.xlsx
df_original = pd.read_excel('/content/sample_data/World Happiness Report  2019.xlsx')
df = df_original.copy() # Criar uma cópia para não alterar o original diretamente

# 2. Selecionar as seis últimas colunas quantitativas
quantitative_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(quantitative_cols) >= 6:
    selected_features = quantitative_cols[-6:]
else:
    selected_features = quantitative_cols

df_selected = df[selected_features].copy()

# --- Handle NaN values before scaling ---
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
if len(df_selected) < initial_rows:
    print(f"Foram removidas {initial_rows - len(df_selected)} linhas devido a valores NaN.")

# --- Check if df_selected is empty after dropping NaNs ---
if df_selected.empty:
    print("Erro: O DataFrame ficou vazio após a remoção de valores NaN. Não há dados para processar.")
else:
    # 3. Padronizar as variáveis
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_selected)

    # 4. Aplicar o método Isolation Forest do PyOD
    clf = IForest(contamination=0.05, random_state=42, n_estimators=100)
    clf.fit(X_scaled)

    # Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
    anomaly_scores = clf.decision_scores_
    predicted_labels = clf.labels_

    # Adicionar os escores e rótulos ao DataFrame original (alinhando pelos índices após dropna)
    df = df.loc[df_selected.index].copy() # Garantir que o df corresponda aos dados usados na detecção
    df['anomaly_score'] = anomaly_scores
    df['is_outlier'] = predicted_labels

    # --- End of replication from OUT15 ---

    # Separar inliers e outliers com base na detecção do iForest
    inliers = df[df['is_outlier'] == 0]
    outliers = df[df['is_outlier'] == 1]

    print(f"Gerando diagramas de dispersão para {len(itertools.combinations(selected_features, 2))} pares de features selecionadas.")

    # Gerar um diagrama de dispersão para cada par de features
    for f1, f2 in itertools.combinations(selected_features, 2):
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers, x=f1, y=f2, color='blue', s=50, alpha=0.7, label='Inliers')
        sns.scatterplot(data=outliers, x=f1, y=f2, color='red', s=100, marker='X', label='Outliers')
        plt.title(f'Diagrama de Dispersão de {f1} vs {f2} com Outliers (iForest)')
        plt.xlabel(f1)
        plt.ylabel(f2)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()


In [ ]:
################################################################################
#   Código OUT2 - box plot de idade na distribuição de idades dos frequentadores
#   assíduos de uma choperia
################################################################################
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Simular a idade dos frequentadres assiduos de uma choperia
#Gerar 250 números aleatórios de uma distribuição Gaussiana
# com média de 40 e desvio padrão de 7
np.random.seed(25)
gausian_data = np.random.normal(loc=40, scale=5, size=250)

#"forçar" a ocorrencia de um outlier:
#Substituir o menor valor da série por 10
min_index = np.argmin(gausian_data)
gausian_data_modified = gausian_data.copy()
gausian_data_modified[min_index] = 10



# 3. Criar um histograma e um boxplot dos dados modificados
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Histograma para os dados modificados
sns.histplot(gausian_data_modified, kde=False, ax=axes[0], color='lightgray')
axes[0].set_title('Histograma das Idades')
axes[0].set_xlabel('idade')
axes[0].set_ylabel('Frequência')

# Box plot para os dados modificados, destacando outliers em vermelho
sns.boxplot(y=gausian_data_modified, ax=axes[1], color='lightgray', flierprops=dict(markerfacecolor='red', marker='o', markeredgecolor='red', markersize=8))
axes[1].set_title('Box Plot das Idades')
axes[1].set_ylabel('idade')

plt.tight_layout() # Ajusta o layout para evitar sobreposição de títulos/rótulos
plt.show()

# Identificar e printar o outlier introduzido
outlier_value = 10.0
outlier_observation_numbers = np.where(gausian_data_modified == outlier_value)[0]



# Identificar todos os outliers com base na regra do box plot (1.5 * IQR)
Q1 = np.percentile(gausian_data_modified, 25)
Q3 = np.percentile(gausian_data_modified, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = gausian_data_modified[(gausian_data_modified < lower_bound) | (gausian_data_modified > upper_bound)]
outlier_indices = np.where((gausian_data_modified < lower_bound) | (gausian_data_modified > upper_bound))[0]

print(f"\nValores Q1: {Q1:.2f}")
print(f"Valores Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Limite Inferior (Q1 - 1.5*IQR): {lower_bound:.2f}")
print(f"Limite Superior (Q3 + 1.5*IQR): {upper_bound:.2f}")

if len(outliers) > 0:
    print(f"\nTotal de outliers detectados (fora dos limites do box plot): {len(outliers)}")
    print("Detalhes dos Outliers:")
    for i in range(len(outliers)):
        print(f"  IDADE: {outliers[i]:.2f}, na observação de número: {outlier_indices[i]}")
else:
    print("\nNão foram encontrados outros outliers fora dos limites do box plot.")

In [ ]:
################################################################################
#   Código OUT3 - geração de dist assimétrica e gráficos
################################################################################
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# vamos gerar uma distribuição assimétrica - qui2 com 2 graus de liberdade
# com 100 valores
np.random.seed(11)
df_chi2 = 2 # graus de liberdade
random_numbers = np.random.chisquare(df_chi2, 100)

# Create a figure with two subplots for histogram and boxplot
fig, axes = plt.subplots(1, 2, figsize=(9, 5))


sns.histplot(random_numbers, kde=False, ax=axes[0], color='lightgray')
axes[0].set_title(f'Histogram of Chi-squared Distribution (df={df_chi2})')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# Boxplot
sns.boxplot(y=random_numbers, ax=axes[1], color='lightgray', flierprops=dict(markerfacecolor='red', marker='o', markeredgecolor='red', markersize=8))
axes[1].set_title(f'Box Plot of Chi-squared Distribution (df={df_chi2})')
axes[1].set_ylabel('Value')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

In [ ]:
###############################################################################
#    Codigo OUT4 -  Geração de base de dados TST01 para ilustração dos métodos
###############################################################################

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
!pip install openpyxl -q

np.random.seed(42)
n_samples = 1000
n_outliers = 60

print("\n--- Manually defining centers for precise separation ---")
manual_centers = np.array([
    [-5, -5],  # Center 1
    [ 5,  5]    # Center 2
])

X_normal,y_normal= make_blobs(
    n_samples=n_samples,
    n_features=2,
    centers=manual_centers,
    cluster_std=1.6,
    random_state=18
)

X_anomalies = np.random.uniform(-10, 10, (n_outliers, 2))
X = np.vstack([X_normal, X_anomalies])
y_true = np.hstack([np.zeros(n_samples - n_outliers), np.ones(n_outliers)])

# Standardize the data (important for Isolation Forest)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Plotting X without differentiating origin
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.7, color='blue') # Plot all points in blue
plt.title('Scatterplot of Generated Data (X) - All Points')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


# Define the output filename
output_filename = 'TST01.xlsx'

# Convert NumPy array X to a pandas DataFrame
df_X = pd.DataFrame(X, columns=['Feature_1', 'Feature_2']) # Assign column names as desired

# Export the DataFrame to Excel
df_X.to_excel(output_filename, index=False) # index=False prevents writing the DataFrame index as a column

print(f"DataFrame successfully exported to '{output_filename}'")
print("You can find the file in the Colab file browser (left sidebar -> folder icon).")

In [ ]:
################################################################################
#   código OUT5 - Análise dos boxplots de x e y em TST01.xlsx
################################################################################
import pandas as pd

# Replace 'your_file.xlsx' with the actual path to your Excel file
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Display the first 5 rows of the DataFrame
display(df.head())

import matplotlib.pyplot as plt
import seaborn as sns
################################################################################
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='Feature_1', y='Feature_2',  s=30, color="gray" )
plt.title('Scatter Plot of Feature_2 vs Feature_1')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.show()
################################################################################
# Create a figure with two subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(6, 4))

# Box plot for 'x'
sns.boxplot(y=df['Feature_1'], ax=axes[0],color='lightgray')
axes[0].set_title('Box Plot of Feature_1')
axes[0].set_ylabel('Feature_1')

# Box plot for 'y'
sns.boxplot(y=df['Feature_2'], ax=axes[1],color='lightgray')
axes[1].set_title('Box Plot of Feature_2')
axes[1].set_ylabel('Feature_2')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

In [ ]:
################################################################################
#  Código OUT6  - cálculo do Z-score modificado
################################################################################
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Simular a idade dos frequentadres assiduos de uma choperia
#Gerar 250 números aleatórios de uma distribuição Gaussiana
# com média de 40 e desvio padrão de 7
np.random.seed(25)
gausian_data = np.random.normal(loc=40, scale=5, size=250)

#"forçar" a ocorrencia de um outlier:
#Substituir o menor valor da série por 10
min_index = np.argmin(gausian_data)
gausian_data_modified = gausian_data.copy()
gausian_data_modified[min_index] = 10

# Aplicar o Z-Score modificado para identificar outliers
median = np.median(gausian_data_modified)
mad = np.median(np.abs(gausian_data_modified - median))

# Evitar divisão por zero se MAD for 0
if mad == 0:
    print("Não é possível calcular o Z-Score modificado, pois o MAD é zero.")
else:
    modified_z_scores = 0.6745 * (gausian_data_modified - median) / mad

    # Definir um limite para outliers (comumente 3.5)
    threshold = 3.5

    # Encontrar os outliers
    outlier_indices_mz = np.where(np.abs(modified_z_scores) > threshold)[0]
    outliers_mz = gausian_data_modified[outlier_indices_mz]

    print(f"\nZ-Score Modificado - Mediana: {median:.2f}")
    print(f"Z-Score Modificado - MAD: {mad:.2f}")
    print(f"Z-Score Modificado - Limite (threshold): {threshold}")

    if len(outliers_mz) > 0:
        print(f"\nTotal de outliers detectados (Z-Score modificado): {len(outliers_mz)}")
        print("Detalhes dos Outliers (Z-Score modificado):")
        for i in range(len(outliers_mz)):
            print(f"  Valor: {outliers_mz[i]:.2f}, na observação de número: {outlier_indices_mz[i]}")
    else:
        print("\nNão foram encontrados outliers usando o Z-Score modificado.")

In [ ]:
################################################################################
#  Código OUT7 - Z-score modificado para uma distribição assimétrica
################################################################################
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# vamos gerar uma distribuição assimétrica - qui2 com 2 graus de liberdade
# com 100 valores
np.random.seed(11)
df_chi2 = 2 # graus de liberdade
random_numbers = np.random.chisquare(df_chi2, 100)

# Aplicar o Z-Score modificado para identificar outliers
median = np.median(random_numbers)
mad = np.median(np.abs(random_numbers - median))

# Evitar divisão por zero se MAD for 0
if mad == 0:
    print("Não é possível calcular o Z-Score modificado, pois o MAD é zero.")
else:
    modified_z_scores = 0.6745 * (random_numbers - median) / mad

    # Definir um limite para outliers (comumente D=3.5)
    threshold = 3.5

    # Encontrar os outliers
    outlier_indices_mz = np.where(np.abs(modified_z_scores) > threshold)[0]
    outliers_mz = random_numbers[outlier_indices_mz]

    print(f"\nZ-Score Modificado - Mediana: {median:.2f}")
    print(f"Z-Score Modificado - MAD: {mad:.2f}")
    print(f"Z-Score Modificado - Limite (threshold): {threshold}")

    if len(outliers_mz) > 0:
        print(f"\nTotal de outliers detectados (Z-Score modificado): {len(outliers_mz)}")
        print("Detalhes dos Outliers (Z-Score modificado):")
        for i in range(len(outliers_mz)):
            print(f"  Valor: {outliers_mz[i]:.2f}, Z-Score: {modified_z_scores[outlier_indices_mz[i]]:.2f}, na observação de número: {outlier_indices_mz[i]}")
    else:
        print("\nNão foram encontrados outliers usando o Z-Score modificado.")

In [ ]:
###############################################################################
#   Código OUT8 -  Geração de base de dados TST02 para ilustração dos métodos
###############################################################################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.covariance import MinCovDet
import scipy.stats

# --- Geração dos Dados Correlacionados Originais ---
# Parâmetros da distribuição
mean = 50
std_dev = 4
correlation = 0.7
num_cases = 100

# Vetor de médias
means = [mean, mean]

# Matriz de covariância
variance = std_dev**2
covariance_xy = correlation * std_dev * std_dev

cov_matrix = [
    [variance, covariance_xy],
    [covariance_xy, variance]
]

# Gerar os dados utilizando numpy.random.multivariate_normal
np.random.seed(11) # Para reprodutibilidade
data = np.random.multivariate_normal(means, cov_matrix, num_cases)

# Criar um DataFrame para melhor visualização
df_correlated = pd.DataFrame(data, columns=['x', 'y'])

print(f"\nVerificação da Correlação original: {df_correlated['x'].corr(df_correlated['y']):.2f}")

# criaçao de outliers
new_observations = pd.DataFrame({
    'x': [50,55, 42.5, 42],
    'y': [62,25, 90, 84]
})

# Concatenar todas as observações
df_correlated_extended = pd.concat([df_correlated, new_observations], ignore_index=True)
print(f"\nTotal de observações no DataFrame estendido: {len(df_correlated_extended)}")

# --- Plotagem do Diagrama de Dispersão ---
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_correlated_extended, x='x', y='y', color='black', s=50, alpha=0.7)
sns.scatterplot(data=new_observations, x='x', y='y', color='red',marker="s", s=50, alpha=0.7)
plt.title('Diagrama de Dispersão de X vs Y ')
plt.xlabel('Variável X')
plt.ylabel('Variável Y')
plt.grid(True, linestyle='--', alpha=0.6)
# Definir o limite do eixo x
plt.xlim(40, 60)
# Definir o limite do eixo y
plt.ylim(0, 100)
plt.show()
df_mcd=df_correlated_extended

# Salva o DataFrame df_mcd em um arquivo Excel
# O arquivo será salvo no ambiente do Colab
output_filename = 'TST02.xlsx' # Adicionado o sufixo '.xlsx'
df_mcd.to_excel(output_filename, index=False)

print(f"DataFrame salvo como '{output_filename}'")

K NN

In [ ]:
# ################################################################################
# #  Código OUT9   kNN para detecção de outliers no arquivo TST01 com k=10
# ################################################################################
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.knn import KNN

# 1. Carregar os dados do arquivo TST01.xlsx
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) #array NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)


# 3. Aplicar o método kNN do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 5%)
knn_algorithm_param = 'auto' # Store the parameter value
clf = KNN(contamination=0.05, algorithm=knn_algorithm_param, n_neighbors=10)
clf.fit(X_scaled)

# 4. Imprimir qual o algoritmo utilizado
print(f"Algoritmo kNN especificado: {knn_algorithm_param}")
print(f"Método de ajuste do NearestNeighbors (escolha 'auto'): {clf.neigh_._fit_method}")


# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores = clf.decision_scores_
predicted_labels = clf.labels_

# Adicionar os escores e rótulos ao DataFrame original ou escalado
df_scaled['anomaly_score'] = anomaly_scores
df_scaled['is_outlier'] = predicted_labels
df['anomaly_score'] = anomaly_scores # Adicionar ao df original para plotagem
df['is_outlier'] = predicted_labels # Adicionar ao df original para plotagem
print(f"O valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")
# Separar inliers e outliers para plotagem
inliers = df[df['is_outlier'] == 0]
outliers = df[df['is_outlier'] == 1]
print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")
# 5. Fazer o diagrama de dispersão marcando os outliers em vermelho com x
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (kNN)')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# 6. Fazer o histograma dos escores de anomalia
plt.figure(figsize=(8, 5))
sns.histplot(anomaly_scores, bins=20, kde=True, color='lightgray')
plt.title('Histograma dos Escores de Anomalia kNN')
plt.xlabel('Escore de Anomalia')
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\nDetalhes dos Outliers detectados (kNN):")
display(outliers.sort_values(by='anomaly_score', ascending=False).head())

################################################################################
# --- Inserindo um novo threshold manual ---
################################################################################
print("\n--- Aplicando um NOVO threshold manual para detecção de outliers ---")
novo_threshold = 0.4  # Defina o valor do seu novo threshold aqui

# Identificar outliers com base no novo threshold
outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

print(f"Novo threshold manual definido: {novo_threshold}")
print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

# Plotar novamente com o novo threshold
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_novo_threshold, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
sns.scatterplot(data=outliers_novo_threshold, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
plt.title(f'Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (Threshold: {novo_threshold})')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados com o NOVO threshold (kNN):")
display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False).head())

In [ ]:
# ################################################################################
# #   Código OUT10   kNN para detecção de outliers no arquivo TST01 com k=30
# ################################################################################
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.knn import KNN

# 1. Carregar os dados do arquivo TST01.xlsx
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) #array NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)


# 3. Aplicar o método kNN do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 5%)
knn_algorithm_param = 'auto' # Store the parameter value
clf = KNN(contamination=0.05, algorithm=knn_algorithm_param, n_neighbors=30)
clf.fit(X_scaled)

# 4. Imprimir qual o algoritmo utilizado
print(f"Algoritmo kNN especificado: {knn_algorithm_param}")
print(f"Método de ajuste do NearestNeighbors (escolha 'auto'): {clf.neigh_._fit_method}")


# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores = clf.decision_scores_
predicted_labels = clf.labels_

# Adicionar os escores e rótulos ao DataFrame original ou escalado
df_scaled['anomaly_score'] = anomaly_scores
df_scaled['is_outlier'] = predicted_labels
df['anomaly_score'] = anomaly_scores # Adicionar ao df original para plotagem
df['is_outlier'] = predicted_labels # Adicionar ao df original para plotagem
print(f"O valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")
# Separar inliers e outliers para plotagem
inliers = df[df['is_outlier'] == 0]
outliers = df[df['is_outlier'] == 1]
print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")
# 5. Fazer o diagrama de dispersão marcando os outliers em vermelho com x
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (kNN)')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# 6. Fazer o histograma dos escores de anomalia
plt.figure(figsize=(8, 5))
sns.histplot(anomaly_scores, bins=20, kde=True, color='lightgray')
plt.title('Histograma dos Escores de Anomalia kNN')
plt.xlabel('Escore de Anomalia')
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\nDetalhes dos Outliers detectados (kNN):")
display(outliers.sort_values(by='anomaly_score', ascending=False).head())

################################################################################
# --- Inserindo um novo threshold manual ---
################################################################################
print("\n--- Aplicando um NOVO threshold manual para detecção de outliers ---")
novo_threshold = 0.5  # Defina o valor do seu novo threshold aqui

# Identificar outliers com base no novo threshold
outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

print(f"Novo threshold manual definido: {novo_threshold}")
print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

# Plotar novamente com o novo threshold
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_novo_threshold, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
sns.scatterplot(data=outliers_novo_threshold, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
plt.title(f'Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (Threshold: {novo_threshold})')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados com o NOVO threshold (kNN):")
display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False).head())

In [ ]:
################################################################################
#   Instalação e atualização de bibliotecas necessárias
################################################################################
!pip install --upgrade pip
!pip install --upgrade pyod --force-reinstall

In [ ]:
################################################################################
#    Código OUT11 - K-NN aplicado ao arquivo WHR com k=10
################################################################################
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Carregar o arquivo Excel
df = pd.read_excel('/content/sample_data/WorldHappinessReport2019.xlsx')

# 2. Selecionar as seis últimas colunas (indicadores de felicidade)
# Certifique-se de que há pelo menos 6 colunas
if df.shape[1] >= 6:
    df_selected_cols = df.iloc[:, -6:]
    print("As seis últimas colunas selecionadas são:")
    print(df_selected_cols.columns.tolist())
else:
    print("O DataFrame tem menos de 6 colunas. Selecionando todas as colunas disponíveis.")
    df_selected_cols = df.copy()
print(df_selected_cols.head())

# Padronizar as variáveis
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_selected_cols)

# Gerar um novo DataFrame com as variáveis padronizadas
df_stand= pd.DataFrame(df_scaled, columns=df_selected_cols.columns)

# Aplicar o método kNN do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 5%)
knn_algorithm_param = 'auto' # Store the parameter value
clf = KNN(contamination=0.05, algorithm=knn_algorithm_param, n_neighbors=10)
clf.fit(df_stand)

# 4. Imprimir qual o algoritmo utilizado
print(f"Algoritmo kNN especificado: {knn_algorithm_param}")
print(f"Método de ajuste do NearestNeighbors (escolha 'auto'): {clf.neigh_._fit_method}")


# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores = clf.decision_scores_
predicted_labels = clf.labels_

# Adicionar os escores e rótulos ao DataFrame original ou escalado
df_stand['anomaly_score'] = anomaly_scores
df_stand['is_outlier'] = predicted_labels
df['anomaly_score'] = anomaly_scores # Adicionar ao df original para plotagem
df['is_outlier'] = predicted_labels # Adicionar ao df original para plotagem
print(f"O valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")
# Separar inliers e outliers para plotagem
inliers = df[df['is_outlier'] == 0]
outliers = df[df['is_outlier'] == 1]
print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")

#Fazer o histograma dos escores de anomalia
plt.figure(figsize=(8, 5))
sns.histplot(anomaly_scores, bins=20, kde=True, color='lightgray')
plt.title('Histograma dos Escores de Anomalia kNN')
plt.xlabel('Escore de Anomalia')
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\nDetalhes dos Outliers detectados (kNN):")
display(outliers.sort_values(by='anomaly_score', ascending=False))

################################################################################
# --- Inserindo um novo threshold manual ---
################################################################################
print("\n--- Aplicando um NOVO threshold manual para detecção de outliers ---")
novo_threshold = 2.2  # Defina o valor do seu novo threshold aqui

# Identificar outliers com base no novo threshold
outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

print(f"Novo threshold manual definido: {novo_threshold}")
print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

print("\nDetalhes dos Outliers detectados com o NOVO threshold (kNN):")
display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False))


In [ ]:
# ################################################################################
# #     Código OUT12 -LOF para detecção de outliers no arquivo TST01 com k=30
# ################################################################################
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.lof import LOF # Importar o algoritmo LOF

# 1. Carregar os dados do arquivo TST01.xlsx
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) #array do NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)


# 3. Aplicar o método LOF do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 5%)
# n_neighbors é o k no k-Nearest Neighbors, usado para calcular a densidade local
clf = LOF(contamination=0.05, n_neighbors=30) # Usar LOF com k=30
clf.fit(X_scaled)

# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores = clf.decision_scores_ # Escores de anomalia do LOF
predicted_labels = clf.labels_       # Rótulos de outlier (0 ou 1)

# Adicionar os escores e rótulos ao DataFrame original
df['anomaly_score'] = anomaly_scores
df['is_outlier'] = predicted_labels

print(f"O valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")

# Separar inliers e outliers para plotagem
inliers = df[df['is_outlier'] == 0]
outliers = df[df['is_outlier'] == 1]
print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")

# 5. Fazer o diagrama de dispersão marcando os outliers em vermelho com X
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (LOF, k=30)')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# 6. Fazer o histograma dos escores de anomalia
plt.figure(figsize=(8, 5))
sns.histplot(anomaly_scores, bins=20, kde=True, color='lightgray')
plt.title('Histograma dos Escores de Anomalia LOF')
plt.xlabel('Escore de Anomalia')
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\nDetalhes dos Outliers detectados (LOF):")
display(outliers.sort_values(by='anomaly_score', ascending=False).head())

################################################################################
# --- Inserindo um novo threshold manual --- (Opcional: ajuste após analisar o histograma)
################################################################################
print("\n--- Aplicando um NOVO threshold manual para detecção de outliers (LOF) ---")
# Sugestão de threshold inicial para LOF é geralmente > 1.0. Ajuste conforme o histograma.
novo_threshold = 1.85 # Você pode ajustar este valor com base na análise do histograma

# Identificar outliers com base no novo threshold
outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

print(f"Novo threshold manual definido: {novo_threshold}")
print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

# Plotar novamente com o novo threshold
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_novo_threshold, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
sns.scatterplot(data=outliers_novo_threshold, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
plt.title(f'Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (LOF, Threshold: {novo_threshold})')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados com o NOVO threshold (LOF):")
display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False).head())

In [ ]:
# ################################################################################
# #  Código OUT13 - LOF para detecção de outliers no arquivo World Happiness Report
# ################################################################################
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.lof import LOF

# 1. Carregar os dados do arquivo World Happiness Report 2019.xlsx
df = pd.read_excel('/content/sample_data/WorldHappinessReport2019.xlsx')

# 2. Selecionar as seis últimas colunas quantitativas
quantitative_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(quantitative_cols) >= 6:
    selected_features = quantitative_cols[-6:]
    print(f"As seis últimas colunas quantitativas selecionadas são: {selected_features}")
else:
    selected_features = quantitative_cols
    print("O DataFrame tem menos de 6 colunas quantitativas. Selecionando todas as disponíveis.")

df_selected = df[selected_features].copy()

# --- Handle NaN values before scaling ---
# Drop rows where any of the selected features have NaN values
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
if len(df_selected) < initial_rows:
    print(f"Foram removidas {initial_rows - len(df_selected)} linhas devido a valores NaN.")
else:
    print("Nenhuma linha com NaN foi removida.")

# --- Check if df_selected is empty after dropping NaNs ---
if df_selected.empty:
    print("Erro: O DataFrame ficou vazio após a remoção de valores NaN. Não há dados para processar.")
else:
    # 3. Padronizar as variáveis
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_selected)
    df_scaled = pd.DataFrame(X_scaled, columns=selected_features)

    # 4. Aplicar o método LOF do PyOD com k=12
    clf = LOF(contamination=0.05, n_neighbors=12)
    clf.fit(X_scaled)

    # Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
    anomaly_scores = clf.decision_scores_
    predicted_labels = clf.labels_

    # Adicionar os escores e rótulos aos DataFrames
    df_selected['anomaly_score'] = anomaly_scores
    df_selected['is_outlier'] = predicted_labels
    # Add to the original df for broader context
    df['anomaly_score'] = anomaly_scores
    df['is_outlier'] = predicted_labels

    print(f"\nO valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")

    # Separar inliers e outliers do DataFrame original 'df'
    inliers = df[df['is_outlier'] == 0]
    outliers = df[df['is_outlier'] == 1]
    print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")

    # 5. Gerar o histograma dos escores de anomalia
    plt.figure(figsize=(8, 5))
    sns.histplot(anomaly_scores, bins=20, kde=True, color='lightgray')
    plt.title('Histograma dos Escores de Anomalia LOF (World Happiness Report)')
    plt.xlabel('Escore de Anomalia')
    plt.ylabel('Frequência')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    print("\nDetalhes dos Outliers detectados (LOF):")
    display(outliers.sort_values(by='anomaly_score', ascending=False)) # Display all columns of outliers

    ################################################################################
    # --- Inserindo um novo threshold manual --- (Opcional: ajuste após analisar o histograma)
    ################################################################################
    print("\n--- Aplicando um NOVO threshold manual para detecção de outliers (LOF) ---")
    # Sugestão de threshold inicial para LOF é geralmente > 1.0. Ajuste este valor com base na análise do histograma
    novo_threshold = 1.5 # Ajuste este valor com base na análise do histograma

    # Identificar outliers com base no novo threshold no DataFrame original 'df'
    outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
    inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

    print(f"Novo threshold manual definido: {novo_threshold}")
    print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

    # Removido a geração do scatter plot com novo threshold
    # if len(selected_features) >= 2:
    #     plt.figure(figsize=(10, 7))
    #     sns.scatterplot(data=inliers_novo_threshold, x=feature1, y=feature2, color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
    #     sns.scatterplot(data=outliers_novo_threshold, x=feature1, y=feature2, color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
    #     plt.title(f'Diagrama de Dispersão de {feature1} vs {feature2} com Outliers (LOF, Threshold: {novo_threshold})')
    #     plt.xlabel(feature1)
    #     plt.ylabel(feature2)
    #     plt.grid(True, linestyle='--', alpha=0.6)
    #     plt.legend()
    #     plt.show()

    print("\nDetalhes dos Outliers detectados com o NOVO threshold (LOF):")
    display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False)) # Display all columns of outliers

In [ ]:
################################################################################
# #  Código OUT14 - iForest para detecção de outliers no arquivo TST01
################################################################################
!pip install --upgrade pyod -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.iforest import IForest # Importar o algoritmo Isolation Forest

# 1. Carregar os dados do arquivo TST01.xlsx
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) #array NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 3. Aplicar o método Isolation Forest do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 5%)
clf = IForest(contamination=0.05, random_state=42) # Usar Isolation Forest
clf.fit(X_scaled)

# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores = clf.decision_scores_ # Escores de anomalia do iForest
predicted_labels = clf.labels_       # Rótulos de outlier (0 ou 1)

# Adicionar os escores e rótulos ao DataFrame original
df['anomaly_score'] = anomaly_scores
df['is_outlier'] = predicted_labels

print(f"O valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")

# Separar inliers e outliers para plotagem
inliers = df[df['is_outlier'] == 0]
outliers = df[df['is_outlier'] == 1]
print(f"Número de outliers detectados : {sum(df['is_outlier'] == 1)}")

# 4. Fazer um gráfico de linhas marcando os escores em ordem decrescente
scores_sorted = pd.Series(anomaly_scores).sort_values(ascending=False).reset_index(drop=True)
plt.figure(figsize=(10, 6))
sns.lineplot(x=scores_sorted.index, y=scores_sorted.values, color='blue')
plt.axhline(y=clf.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf.threshold_:.4f}')
plt.title('Escores de Anomalia do iForest em Ordem Decrescente')
plt.xlabel('Índice da Observação (ordenado por escore)')
plt.ylabel('Escore de Anomalia')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 5. Fazer o diagrama de dispersão marcando os outliers em vermelho com X (Feature_1 vs Feature_2)
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (Automático)')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (iForest - Threshold Automático)')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# 6. Fazer o gráfico de dispersão dos escores vs. Feature_1, marcando outliers com uma cruz vermelha
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers, x='Feature_1', y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers, x='Feature_1', y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Automático)')
plt.axhline(y=clf.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf.threshold_:.4f}')
plt.title('Escore de Anomalia vs. Feature_1 com Outliers (iForest - Threshold Automático)')
plt.xlabel('Feature_1')
plt.ylabel('Escore de Anomalia')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados (iForest - Threshold Automático):")
display(outliers.sort_values(by='anomaly_score', ascending=False).head())

################################################################################
# --- Inserindo um novo threshold manual ---
################################################################################
print("\n--- Aplicando um NOVO threshold manual para detecção de outliers (iForest) ---")
novo_threshold = 0.02  # Defina o valor do seu novo threshold aqui

# Identificar outliers com base no novo threshold
outliers_novo_threshold = df[df['anomaly_score'] > novo_threshold]
inliers_novo_threshold = df[df['anomaly_score'] <= novo_threshold]

print(f"Novo threshold manual definido: {novo_threshold}")
print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

# Plotar novamente o diagrama de dispersão com o novo threshold
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_novo_threshold, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
sns.scatterplot(data=outliers_novo_threshold, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
plt.title(f'Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (iForest - Threshold: {novo_threshold})')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# Plotar novamente o gráfico de dispersão dos escores vs. Feature_1 com o novo threshold
plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_novo_threshold, x='Feature_1', y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
sns.scatterplot(data=outliers_novo_threshold, x='Feature_1', y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
plt.axhline(y=novo_threshold, color='red', linestyle='--', label=f'Threshold Manual: {novo_threshold:.4f}')
plt.title(f'Escore de Anomalia vs. Feature_1 com Outliers (iForest - Threshold: {novo_threshold})')
plt.xlabel('Feature_1')
plt.ylabel('Escore de Anomalia')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados com o NOVO threshold (iForest):")
display(outliers_novo_threshold.sort_values(by='anomaly_score', ascending=False).head())

In [ ]:
################################################################################
# #  Código OUT15 - iForest para detecção de outliers no arquivo World Happiness Report
################################################################################
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.iforest import IForest # Importar o algoritmo Isolation Forest

# 1. Carregar os dados do arquivo World Happiness Report 2019.xlsx
df_original = pd.read_excel('/content/sample_data/WorldHappinessReport2019.xlsx')
df = df_original.copy() # Criar uma cópia para não alterar o original diretamente

# 2. Selecionar as seis últimas colunas quantitativas
quantitative_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(quantitative_cols) >= 6:
    selected_features = quantitative_cols[-6:]
    print(f"As seis últimas colunas quantitativas selecionadas são: {selected_features}")
elif len(quantitative_cols) > 0:
    selected_features = quantitative_cols
    print(f"O DataFrame tem menos de 6 colunas quantitativas. Selecionando todas as {len(selected_features)} disponíveis: {selected_features}")
else:
    print("Erro: Não há colunas quantitativas no DataFrame para análise.")
    exit() # Interromper a execução se não houver features

df_selected = df[selected_features].copy()

# --- Handle NaN values before scaling ---
# Drop rows where any of the selected features have NaN values
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
if len(df_selected) < initial_rows:
    print(f"Foram removidas {initial_rows - len(df_selected)} linhas devido a valores NaN.")
else:
    print("Nenhuma linha com NaN foi removida.")

# --- Check if df_selected is empty after dropping NaNs ---
if df_selected.empty:
    print("Erro: O DataFrame ficou vazio após a remoção de valores NaN. Não há dados para processar.")
else:
    # 3. Padronizar as variáveis
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_selected)
    df_scaled = pd.DataFrame(X_scaled, columns=selected_features)

    # 4. Aplicar o método Isolation Forest do PyOD
    # contamination é a proporção de outliers esperada no dataset (ex: 5%)
    clf = IForest(contamination=0.05, random_state=42, n_estimators=100) # Usar Isolation Forest com n_estimators=100
    clf.fit(X_scaled)

    # Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
    anomaly_scores = clf.decision_scores_ # Escores de anomalia do iForest
    predicted_labels = clf.labels_       # Rótulos de outlier (0 ou 1)

    # Adicionar os escores e rótulos ao DataFrame original (alinhando pelos índices após dropna)
    df_selected['anomaly_score'] = anomaly_scores
    df_selected['is_outlier'] = predicted_labels

    # Adicionar ao DataFrame original completo para referência
    df = df.loc[df_selected.index].copy() # Garantir que o df corresponda aos dados usados na detecção
    df['anomaly_score'] = anomaly_scores
    df['is_outlier'] = predicted_labels


    print(f"\nO valor de corte (threshold) para os escores de anomalia é: {clf.threshold_:.4f}")
    display(df['anomaly_score'].sort_values(ascending=False))
    # Separar inliers e outliers para plotagem
    inliers = df_selected[df_selected['is_outlier'] == 0]
    outliers = df_selected[df_selected['is_outlier'] == 1]
    print(f"Número de outliers detectados : {sum(df_selected['is_outlier'] == 1)}")

    # 5. Fazer um gráfico de linhas marcando os escores em ordem decrescente
    scores_sorted = pd.Series(anomaly_scores).sort_values(ascending=False).reset_index(drop=True)
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=scores_sorted.index, y=scores_sorted.values, color='blue')
    plt.axhline(y=clf.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf.threshold_:.4f}')
    plt.title('Escores de Anomalia do iForest em Ordem Decrescente (World Happiness Report)')
    plt.xlabel('Índice da Observação (ordenado por escore)')
    plt.ylabel('Escore de Anomalia')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    # 6. Fazer o diagrama de dispersão marcando os outliers em vermelho com X
    # Escolher as duas primeiras features para o scatter plot, se existirem
    if len(selected_features) >= 2:
        feature1 = selected_features[0]
        feature2 = selected_features[1]
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers, x=feature1, y=feature2, color='blue', s=50, alpha=0.7, label='Inliers')
        sns.scatterplot(data=outliers, x=feature1, y=feature2, color='red', s=100, marker='X', label='Outliers (Automático)')
        plt.title(f'Diagrama de Dispersão de {feature1} vs {feature2} com Outliers (iForest - Threshold Automático)')
        plt.xlabel(feature1)
        plt.ylabel(feature2)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()

        # 7. Fazer o gráfico de dispersão dos escores vs. uma feature, marcando outliers com uma cruz vermelha
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers, x=feature1, y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers')
        sns.scatterplot(data=outliers, x=feature1, y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Automático)')
        plt.axhline(y=clf.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf.threshold_:.4f}')
        plt.title(f'Escore de Anomalia vs. {feature1} com Outliers (iForest - Threshold Automático)')
        plt.xlabel(feature1)
        plt.ylabel('Escore de Anomalia')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()
    elif len(selected_features) == 1:
        feature1 = selected_features[0]
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers, x=feature1, y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers')
        sns.scatterplot(data=outliers, x=feature1, y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Automático)')
        plt.axhline(y=clf.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf.threshold_:.4f}')
        plt.title(f'Escore de Anomalia vs. {feature1} com Outliers (iForest - Threshold Automático)')
        plt.xlabel(feature1)
        plt.ylabel('Escore de Anomalia')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()
    else:
        print("Não há features suficientes para gerar diagramas de dispersão.")

    print("\nDetalhes dos Outliers detectados (iForest - Threshold Automático):")
    display(df_original.loc[outliers.index].assign(anomaly_score=outliers['anomaly_score']).sort_values(by='anomaly_score', ascending=False))

    ################################################################################
    # --- Inserindo um novo threshold manual ---
    ################################################################################
    print("\n--- Aplicando um NOVO threshold manual para detecção de outliers (iForest) ---")
    # Ajuste este valor com base na análise do gráfico de linha dos escores de anomalia
    novo_threshold = 0.01  # Sugestão de valor, ajuste conforme a distribuição dos escores

    # Identificar outliers com base no novo threshold
    outliers_novo_threshold = df_selected[df_selected['is_outlier'] == 1] # Filter outliers based on original prediction

    # Now identify based on the new manual threshold. Note: 'anomaly_score' is already in df_selected
    outliers_novo_threshold = df_selected[df_selected['anomaly_score'] > novo_threshold]
    inliers_novo_threshold = df_selected[df_selected['anomaly_score'] <= novo_threshold]

    print(f"Novo threshold manual definido: {novo_threshold}")
    print(f"Número de outliers detectados com o novo threshold: {len(outliers_novo_threshold)}")

    # Plotar novamente o diagrama de dispersão com o novo threshold
    if len(selected_features) >= 2:
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers_novo_threshold, x=feature1, y=feature2, color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
        sns.scatterplot(data=outliers_novo_threshold, x=feature1, y=feature2, color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
        plt.title(f'Diagrama de Dispersão de {feature1} vs {feature2} com Outliers (iForest - Threshold: {novo_threshold})')
        plt.xlabel(feature1)
        plt.ylabel(feature2)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()

        # Plotar novamente o gráfico de dispersão dos escores vs. uma feature com o novo threshold
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers_novo_threshold, x=feature1, y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
        sns.scatterplot(data=outliers_novo_threshold, x=feature1, y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
        plt.axhline(y=novo_threshold, color='red', linestyle='--', label=f'Threshold Manual: {novo_threshold:.4f}')
        plt.title(f'Escore de Anomalia vs. {feature1} com Outliers (iForest - Threshold: {novo_threshold})')
        plt.xlabel(feature1)
        plt.ylabel('Escore de Anomalia')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()
    elif len(selected_features) == 1:
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=inliers_novo_threshold, x=feature1, y='anomaly_score', color='blue', s=50, alpha=0.7, label='Inliers (Novo Threshold)')
        sns.scatterplot(data=outliers_novo_threshold, x=feature1, y='anomaly_score', color='red', s=100, marker='X', label='Outliers (Novo Threshold)')
        plt.axhline(y=novo_threshold, color='red', linestyle='--', label=f'Threshold Manual: {novo_threshold:.4f}')
        plt.title(f'Escore de Anomalia vs. {feature1} com Outliers (iForest - Threshold: {novo_threshold})')
        plt.xlabel(feature1)
        plt.ylabel('Escore de Anomalia')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.show()

    print("\nDetalhes dos Outliers detectados com o NOVO threshold (iForest):")
    display(df_original.loc[outliers_novo_threshold.index].assign(anomaly_score=outliers_novo_threshold['anomaly_score']).sort_values(by='anomaly_score', ascending=False))

In [ ]:
# As instalações do pip devem ser feitas em uma célula separada para garantir que o ambiente seja atualizado corretamente.
# Removendo a instalação do pip daqui, assumindo que já foi feita.
!pip install --upgrade pip
!pip install --upgrade pyod --force-reinstall

In [ ]:
###############################################################################
#   Código OUT15 - Aplicação de ABOD ao arquivo TST01
###############################################################################

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.abod import ABOD
import time

# As instalações do pip devem ser feitas em uma célula separada para garantir que o ambiente seja atualizado corretamente.
# Removendo a instalação do pip daqui, assumindo que já foi feita.
# !pip install --upgrade pip
# !pip install --upgrade pyod --force-reinstall

# 1. Carregar os dados do arquivo TST01.xlsx
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # array NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# --- ABOD (Angle-based Outlier Detection) ---
print("\n--- Executando ABOD ---")
start_time_abod = time.time()

clf_abod = ABOD(contamination=0.05)
clf_abod.fit(X_scaled)

end_time_abod = time.time()
time_taken_abod = end_time_abod - start_time_abod
print(f"Tempo de processamento (ABOD): {time_taken_abod:.4f} segundos")

anomaly_scores_abod = clf_abod.decision_scores_
predicted_labels_abod = clf_abod.labels_

df_abod = df.copy() # Usar uma cópia para não interferir nos resultados do FastABOD
df_abod['anomaly_score'] = anomaly_scores_abod
df_abod['is_outlier'] = predicted_labels_abod

print(f"O valor de corte (threshold) para os escores de anomalia (ABOD) é: {clf_abod.threshold_:.4f}")

inliers_abod = df_abod[df_abod['is_outlier'] == 0]
outliers_abod = df_abod[df_abod['is_outlier'] == 1]
print(f"Número de outliers detectados (ABOD): {len(outliers_abod)}")

plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_abod, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers_abod, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (ABOD)')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (ABOD)')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados (ABOD):")
display(outliers_abod.sort_values(by='anomaly_score', ascending=False).head())



In [ ]:
###############################################################################
#  Código OUT16 -  Aplicação de ABOD ao arquivo TST01 com method='fast'
###############################################################################

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.abod import ABOD
import time

# 1. Carregar os dados do arquivo TST01.xlsx (recarregado para garantir que df esteja atualizado)
df = pd.read_excel('/content/sample_data/TST01.xlsx')

# Selecionar as features para detecção de outliers
X = df[['Feature_1', 'Feature_2']]

# 2. Padronizar as variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # array NumPy

# Criar um DataFrame a partir dos dados padronizados para facilitar a visualização
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# --- ABOD (Angle-based Outlier Detection) com method='fast' ---
print("\n--- Executando ABOD com method='fast' ---")
start_time_abod_fast = time.time()

clf_abod_fast = ABOD(contamination=0.05, method='fast', n_neighbors=30)
clf_abod_fast.fit(X_scaled)

end_time_abod_fast = time.time()
time_taken_abod_fast = end_time_abod_fast - start_time_abod_fast
print(f"Tempo de processamento (ABOD com method='fast'): {time_taken_abod_fast:.4f} segundos")

anomaly_scores_abod_fast = clf_abod_fast.decision_scores_
predicted_labels_abod_fast = clf_abod_fast.labels_

df_abod_fast = df.copy()
df_abod_fast['anomaly_score'] = anomaly_scores_abod_fast
df_abod_fast['is_outlier'] = predicted_labels_abod_fast

print(f"O valor de corte (threshold) para os escores de anomalia (ABOD com method='fast') é: {clf_abod_fast.threshold_:.4f}")

inliers_abod_fast = df_abod_fast[df_abod_fast['is_outlier'] == 0]
outliers_abod_fast = df_abod_fast[df_abod_fast['is_outlier'] == 1]
print(f"Número de outliers detectados (ABOD com method='fast'): {len(outliers_abod_fast)}")

plt.figure(figsize=(10, 7))
sns.scatterplot(data=inliers_abod_fast, x='Feature_1', y='Feature_2', color='blue', s=50, alpha=0.7, label='Inliers')
sns.scatterplot(data=outliers_abod_fast, x='Feature_1', y='Feature_2', color='red', s=100, marker='X', label='Outliers (ABOD com method=\'fast\')')
plt.title('Diagrama de Dispersão de Feature_1 vs Feature_2 com Outliers (ABOD com method=\'fast\')')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nDetalhes dos Outliers detectados (ABOD com method='fast'):")
display(outliers_abod_fast.sort_values(by='anomaly_score', ascending=False).head())

In [ ]:
###############################################################################
#  Código OUT17 -  estudo de caso ATAKIM - parte 1 - análises univariadas
###############################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar o arquivo ATAKIM.xlsx para um DataFrame
df = pd.read_excel('/content/sample_data/ATAKIM.xlsx')
print("DataFrame Carregado:")
display(df.head())

# Identificar variáveis qualitativas e quantitativas
# Excluir 'IDENTIDADE' das variáveis qualitativas conforme solicitado
qualitative_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if 'IDENTIDADE' in qualitative_cols:
    qualitative_cols.remove('IDENTIDADE')

quantitative_cols = df.select_dtypes(include=np.number).columns.tolist()

print(f"\nVariáveis Qualitativas (excluindo IDENTIDADE): {qualitative_cols}")
print(f"Variáveis Quantitativas: {quantitative_cols}")

# --- Análise de Variáveis Qualitativas (Tabelas de Frequência Absoluta) ---
if qualitative_cols:
    print("\n--- Tabelas de Frequência Absoluta para Variáveis Qualitativas ---")
    for col in qualitative_cols:
        print(f"\nVariável: {col}")
        display(df[col].value_counts().to_frame(name='Frequência Absoluta'))
else:
    print("\nNão foram encontradas variáveis qualitativas (exceto IDENTIDADE) para análise.")

# --- Análise de Variáveis Quantitativas (Histograma e Box Plot) ---
if quantitative_cols:
    print("\n--- Análise para Variáveis Quantitativas (Histograma e Box Plot) ---")

    for col in quantitative_cols:
        print(f"\nAnalisando a variável: {col}")

        # Removendo NaNs para o cálculo da mediana e MAD
        col_data_clean = df[col].dropna()

        if col_data_clean.empty:
            print(f"Não há dados válidos para a variável '{col}'. Ignorando.")
            continue

        # Imprimir estatísticas de percentis
        print(f"Estatísticas para '{col}':")
        print(f"  Mínimo: {np.min(col_data_clean):.2f}")
        print(f"  P25 (Primeiro Quartil): {np.percentile(col_data_clean, 25):.2f}")
        print(f"  P50 (Mediana): {np.percentile(col_data_clean, 50):.2f}")
        print(f"  P75 (Terceiro Quartil): {np.percentile(col_data_clean, 75):.2f}")
        print(f"  P90: {np.percentile(col_data_clean, 90):.2f}")
        print(f"  P99: {np.percentile(col_data_clean, 99):.2f}")
        print(f"  Máximo: {np.percentile(col_data_clean, 100):.2f}")

        # Histograma e Box Plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Histograma
        sns.histplot(col_data_clean, ax=axes[0], color='lightgray',bins =25)
        axes[0].set_title(f'Histograma de {col}')
        axes[0].set_xlabel(col)
        axes[0].set_ylabel('Frequência')

        # Box Plot
        sns.boxplot(y=col_data_clean, ax=axes[1], color='lightgray',
                    flierprops=dict(markerfacecolor='red', marker='o', markeredgecolor='red', markersize=4))
        axes[1].set_title(f'Box Plot de {col}')
        axes[1].set_ylabel(col)
        plt.grid("lightgray")
        plt.tight_layout()
        plt.show()

    print("\n--- Análise da variável 'TEMPCLI' após eliminação de observações > 13 ---")
    df1 = df[df['TEMPCLI'] <= 13].copy() # Crie df1 filtrando TEMPCLI
    df1.shape

    # Dados limpos de TEMPCLI em df1
    tempcli_data_clean_df1 = df1['TEMPCLI'].dropna()

    if tempcli_data_clean_df1.empty:
        print(f"Não há dados válidos para a variável 'TEMPCLI' em df1. Ignorando.")
    else:
        # Imprimir estatísticas de percentis para TEMPCLI em df1
        print(f"Estatísticas para 'TEMPCLI' em df1:")
        print(f"  Mínimo: {np.min(tempcli_data_clean_df1):.2f}")
        print(f"  P25 (Primeiro Quartil): {np.percentile(tempcli_data_clean_df1, 25):.2f}")
        print(f"  P50 (Mediana): {np.percentile(tempcli_data_clean_df1, 50):.2f}")
        print(f"  P75 (Terceiro Quartil): {np.percentile(tempcli_data_clean_df1, 75):.2f}")
        print(f"  P90: {np.percentile(tempcli_data_clean_df1, 90):.2f}")
        print(f"  P99: {np.percentile(tempcli_data_clean_df1, 99):.2f}")
        print(f"  Máximo: {np.percentile(tempcli_data_clean_df1, 100):.2f}")

        # Histograma e Box Plot para TEMPCLI em df1
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Histograma
        sns.histplot(tempcli_data_clean_df1, ax=axes[0], color='lightgray', bins=25)
        axes[0].set_title('Histograma de TEMPCLI (df1)')
        axes[0].set_xlabel('TEMPCLI')
        axes[0].set_ylabel('Frequência')

        # Box Plot
        sns.boxplot(y=tempcli_data_clean_df1, ax=axes[1], color='lightgray',
                    flierprops=dict(markerfacecolor='red', marker='o', markeredgecolor='red', markersize=4))
        axes[1].set_title('Box Plot de TEMPCLI (df1)')
        axes[1].set_ylabel('TEMPCLI')
        plt.grid("lightgray")
        plt.tight_layout()
        plt.show()
else:
    print("\nNão foram encontradas variáveis quantitativas para análise.")

In [ ]:
# ################################################################################
# #  Código OUT18 -  Estudo de caso ATAKIM - parte 2 - LOF
# ##############################################################################
## Instalar a biblioteca Gower, se necessário
!pip install gower -q
!pip install --upgrade pyod -q

import pandas as pd
import numpy as np
import time # Importar a biblioteca time
from sklearn.preprocessing import StandardScaler
from pyod.models.lof import LOF
from gower import gower_matrix

# Carregar o arquivo ATAKIM.xlsx para um DataFrame
df = pd.read_excel('/content/sample_data/ATAKIM.xlsx')

# Filtrar df para criar df1
df1 = df[df['TEMPCLI'] <= 13].copy()

print(f"Dimensões de df1 após o filtro: {df1.shape}")
display(df1.head())

# --- 1. Separar Identidade e Identificar Variáveis ---
identidades = df1['IDENTIDADE']
df1_features = df1.drop(columns=['IDENTIDADE']).copy()

quantitative_cols = df1_features.select_dtypes(include=np.number).columns.tolist()
qualitative_cols = df1_features.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nVariáveis Qualitativas para One-Hot Encoding (N dummies): {qualitative_cols}")
print(f"Variáveis Quantitativas para Padronização: {quantitative_cols}")

# --- 2. Pré-processamento: Padronização para Quantitativas e One-Hot Encoding (N dummies) para Qualitativas ---

# 2a. Padronizar as variáveis quantitativas
scaler = StandardScaler()
df1_num_scaled = pd.DataFrame(scaler.fit_transform(df1_features[quantitative_cols]),
                              columns=[f"num__{col}" for col in quantitative_cols],
                              index=df1_features.index)

# 2b. Aplicar One-Hot Encoding com drop_first=False às variáveis qualitativas usando pandas (N dummies)
df1_cat_dummies = pd.get_dummies(df1_features[qualitative_cols], drop_first=False, dtype=int)
# Renomear as colunas para clareza, adicionando um prefixo 'cat__' para diferenciá-las das numéricas
df1_cat_dummies.columns = [f"cat__{col}" for col in df1_cat_dummies.columns]

# 2c. Concatenar as features processadas (numéricas padronizadas e categóricas dummy N)
df1_preprocessed_features = pd.concat([df1_num_scaled, df1_cat_dummies], axis=1)

print("\nDataFrame com features pré-processadas (padronizadas e OHE com N dummies):")
display(df1_preprocessed_features.head())
display(df1_preprocessed_features.info())

# --- 3. Calcular a Matriz de Distância de Gower ---
# A matriz de Gower será calculada sobre as features já pré-processadas, que são todas numéricas.
print("\nCalculando Matriz de Distância de Gower sobre as features pré-processadas...")
start_time_gower = time.time() # Início da contagem de tempo para Gower

df_gower = gower_matrix(df1_preprocessed_features)

end_time_gower = time.time() # Fim da contagem de tempo para Gower
time_taken_gower = end_time_gower - start_time_gower
print(f"Tempo de processamento (Matriz de Gower): {time_taken_gower:.4f} segundos")
print(f"Dimensões da Matriz de Distância de Gower: {df_gower.shape}")

# --- 4. Aplicar LOF com a Matriz de Distância de Gower ---
print("\nAplicando LOF com k=50 e contamination=0.005...")
start_time_lof = time.time() # Início da contagem de tempo para LOF

clf_lof_gower = LOF(n_neighbors=50, contamination=0.005, metric='precomputed')
clf_lof_gower.fit(df_gower)

end_time_lof = time.time() # Fim da contagem de tempo para LOF
time_taken_lof = end_time_lof - start_time_lof
print(f"Tempo de processamento (LOF com Gower): {time_taken_lof:.4f} segundos")

anomaly_scores_lof_gower = clf_lof_gower.decision_scores_
predicted_labels_lof_gower = clf_lof_gower.labels_

# Adicionar escores e rótulos ao DataFrame original df1
df1['anomaly_score_lof_gower'] = anomaly_scores_lof_gower
df1['is_outlier_lof_gower'] = predicted_labels_lof_gower

print(f"O valor de corte (threshold) para os escores de anomalia LOF é: {clf_lof_gower.threshold_:.4f}")

# --- 5. Imprimir a identidade dos outliers ---
outliers_lof_gower = df1[df1['is_outlier_lof_gower'] == 1].copy()

if not outliers_lof_gower.empty:
    print(f"\nTotal de outliers detectados pelo LOF (Gower, k=50, contamination=0.005): {len(outliers_lof_gower)}")
    print("\nDetalhes dos Outliers (ordenados por IDENTIDADE):")
    outlier_identities_list_LOF = outliers_lof_gower['IDENTIDADE'].sort_values().tolist()
    print(outlier_identities_list_LOF )
else:
    print("\nNenhum outlier detectado pelo LOF (Gower, k=50, contamination=0.005).")
    outlier_identities_list_LOF  = []

    # 6. Fazer um gráfico de linhas marcando os escores em ordem decrescente
scores_sorted_lof = pd.Series(anomaly_scores_lof_gower).sort_values(ascending=False).reset_index(drop=True)
plt.figure(figsize=(10, 6))
sns.lineplot(x=scores_sorted_lof.index, y=scores_sorted_lof.values, color='blue')
plt.axhline(y=clf_lof_gower.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf_lof_gower.threshold_:.4f}')
plt.title('Escores de Anomalia do LOF (Gower) em Ordem Decrescente')
plt.xlabel('Índice da Observação (ordenado por escore)')
plt.ylabel('Escore de Anomalia')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# ##############################################################################
# #  Código OUT19 -  Estudo de caso ATAKIM - parte 3 - iForest
# ##############################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pyod.models.iforest import IForest

# df1 e df1_preprocessed_features são gerados no Código OUT18 (rBwUJTC2K74S)
# Certifique-se de que o Código OUT18 foi executado antes deste.

# 1. Aplicar o método Isolation Forest do PyOD
# contamination é a proporção de outliers esperada no dataset (ex: 0.005)
print("\nAplicando Isolation Forest com contamination=0.005...")

# n_estimators é um parâmetro importante para iForest. Usar 100 como padrão.
clf_iforest = IForest(contamination=0.005, random_state=11, n_estimators=200)
clf_iforest.fit(df1_preprocessed_features)

# Obter escores de anomalia e rótulos de previsão (0: inlier, 1: outlier)
anomaly_scores_iforest = clf_iforest.decision_scores_
predicted_labels_iforest = clf_iforest.labels_

# Adicionar escores e rótulos ao DataFrame original df1 (que contém IDENTIDADE)
df1['anomaly_score_iforest'] = anomaly_scores_iforest
df1['is_outlier_iforest'] = predicted_labels_iforest

print(f"\nO valor de corte (threshold) para os escores de anomalia do iForest é: {clf_iforest.threshold_:.4f}")

# Separar inliers e outliers do DataFrame df1
outliers_iforest = df1[df1['is_outlier_iforest'] == 1].copy()

if not outliers_iforest.empty:
    print(f"\nTotal de outliers detectados pelo iForest (contamination=0.005): {len(outliers_iforest)}")
    print("\nDetalhes dos Outliers (ordenados por escore de anomalia decrescente):")
    display(outliers_iforest.sort_values(by='anomaly_score_iforest', ascending=False))

    # Gerar a lista de identidades dos outliers, ordenada pela IDENTIDADE em ordem crescente
    outlier_identities_list_IForest = outliers_iforest.sort_values(by='IDENTIDADE', ascending=True)['IDENTIDADE'].tolist()
    print("\nLista de IDENTIDADE dos Outliers (ordenada por IDENTIDADE crescente):")
    print(outlier_identities_list_IForest)
else:
    print("\nNenhum outlier detectado pelo iForest (contamination=0.005).")
    outlier_identities_list_IForest = []

# 2. Fazer um gráfico de linhas marcando os escores em ordem decrescente
scores_sorted_iforest = pd.Series(anomaly_scores_iforest).sort_values(ascending=False).reset_index(drop=True)
plt.figure(figsize=(10, 6))
sns.lineplot(x=scores_sorted_iforest.index, y=scores_sorted_iforest.values, color='blue')
plt.axhline(y=clf_iforest.threshold_, color='red', linestyle='--', label=f'Threshold Automático: {clf_iforest.threshold_:.4f}')
plt.title('Escores de Anomalia do iForest em Ordem Decrescente (ATAKIM)')
plt.xlabel('Índice da Observação (ordenado por escore)')
plt.ylabel('Escore de Anomalia')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# ##############################################################################
# #  Código OUT20 -  Estudo de caso ATAKIM - parte 4 - comparação dos outliers
# ##############################################################################
# Converter as listas para conjuntos para facilitar as operações de comparação
set_lof_outliers = set(outlier_identities_list_LOF)
set_iforest_outliers = set(outlier_identities_list_IForest)

# Pontos em comum (interseção)
common_outliers = set_lof_outliers.intersection(set_iforest_outliers)

# Pontos que são outliers para LOF mas não para iForest
lof_only_outliers = set_lof_outliers.difference(set_iforest_outliers)

# Pontos que são outliers para iForest mas não para LOF
iforest_only_outliers = set_iforest_outliers.difference(set_lof_outliers)

print(f"\nTotal de outliers detectados pelo LOF: {len(set_lof_outliers)}")
print(f"Total de outliers detectados pelo iForest: {len(set_iforest_outliers)}")

print(f"\nOutliers em comum (detectados por ambos os métodos): {len(common_outliers)}")
if common_outliers:
    print("IDs dos Outliers em comum:")
    print(sorted(list(common_outliers)))
else:
    print("Não há outliers em comum entre os dois métodos.")

print(f"\nOutliers exclusivos do LOF: {len(lof_only_outliers)}")
if lof_only_outliers:
    print("IDs dos Outliers exclusivos do LOF:")
    print(sorted(list(lof_only_outliers)))
else:
    print("Não há outliers exclusivos do LOF.")

print(f"\nOutliers exclusivos do iForest: {len(iforest_only_outliers)}")
if iforest_only_outliers:
    print("IDs dos Outliers exclusivos do iForest:")
    print(sorted(list(iforest_only_outliers)))
else:
    print("Não há outliers exclusivos do iForest.")

print(f"\nTotal de pontos que diferem (LOF ou iForest, mas não ambos): {len(lof_only_outliers) + len(iforest_only_outliers)}")